# Core Demo: spec-driven conversion

This notebook profiles a bag, builds a reusable single-trigger conversion spec, runs the converter, and inspects the emitted dataset, manifest, and report files.


In [ ]:
import sys
from pathlib import Path

demo_dir = Path.cwd()
repo_root = demo_dir.parent if demo_dir.name == "demo" else demo_dir
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from hephaes import (
    Converter,
    LabelSpec,
    Profiler,
    build_doom_ros_train_py_compatible,
    build_single_trigger_sensor_log_template,
    stream_tfrecord_rows,
)

bag_files = [
    str(Path("input") / "ros2.mcap"),
]
output_root = Path("./output")


## 1. Profile the bag to get metadata and topics


In [ ]:
topics = []

if not bag_files:
    print("No bag files configured.")
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for topic in meta.topics:
            print(f"  - {topic.name} ({topic.message_type}): {topic.message_count} messages, {topic.rate_hz} Hz")
        print("-" * 40)

topic_names = [topic.name for topic in topics]
print("Discovered topics:", topic_names)


## 2. Build a reusable single-trigger template

The generic template uses the first discovered topic as the trigger and treats the next few topics as optional joins.
You can specialize it with `model_copy(update={...})` to add labels or replace feature transforms.


In [ ]:
if not topic_names:
    raise RuntimeError("The demo bag did not expose any topics to build a template from.")

demo_spec = build_single_trigger_sensor_log_template(
    trigger_topic=topic_names[0],
    join_topics=topic_names[1:4],
)

primary_feature = next(iter(demo_spec.features))
demo_spec = demo_spec.model_copy(
    update={
        "labels": LabelSpec(primary=primary_feature),
        "validation": demo_spec.validation.model_copy(update={"preview": True}),
    }
)

doom_spec = build_doom_ros_train_py_compatible()
print("Generic schema:", demo_spec.schema.model_dump())
print("Generic features:", list(demo_spec.features))
print("Generic label:", demo_spec.labels.primary if demo_spec.labels is not None else None)
print("Doom preset:", doom_spec.schema.model_dump())


## 3. Convert the bag with the spec-driven converter


In [ ]:
spec_output_dir = output_root / "spec_demo"
spec_files = Converter(
    bag_files,
    None,
    spec_output_dir,
    spec=demo_spec,
    max_workers=1,
).convert()

spec_files


## 4. Inspect the emitted rows and audit artifacts


In [ ]:
output_tfrecord = spec_files[0]
manifest_path = output_tfrecord.with_name(f"{output_tfrecord.stem}.manifest.json")
report_path = output_tfrecord.with_name(f"{output_tfrecord.stem}.report.md")

print("Dataset:", output_tfrecord)
print("Manifest:", manifest_path)
print("Report:", report_path)

for i, row in enumerate(stream_tfrecord_rows(output_tfrecord)):
    print(row)
    if i + 1 >= 5:
        break


## 5. Override a built-in preset

The Doom preset is meant to be a starting point. You can keep the contract and still adjust labels, validation, or feature transforms for a new experiment.


In [ ]:
doom_variant = doom_spec.model_copy(
    update={
        "labels": LabelSpec(primary="buttons"),
        "validation": doom_spec.validation.model_copy(update={"preview": True}),
    }
)

print("Variant schema:", doom_variant.schema.model_dump())
print("Variant output:", doom_variant.output.model_dump())
print("To change a feature pipeline, replace one entry in `doom_variant.features` with `model_copy(update={"transforms": [...]})`.")
